
## Problem Statement

Develop a simple Retrieval-Augmented Generation (RAG) system to answer questions from custom documents. Build a pipeline that retrieves relevant information from a document and uses a language model to generate answers.


## Step 1: Imports

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers import BM25Retriever
import time

print("Imports successful")

C:\Users\LOQ\AppData\Local\Temp\ipykernel_5200\4258358230.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Imports successful


## Step 2: Document Ingestion

In [2]:
pdf_path = r"C:\Users\LOQ\Documents\GitHub\CSI_INTERNSHIP_2026\WEEK_7\unit4 dl.pdf"  

loader = PyPDFLoader(pdf_path)
documents = loader.load()
print(f"Loaded {len(documents)} page(s)")
print(documents[0].page_content[:500])  

Loaded 61 page(s)
CSN 342
Deep Learning
1
Faculty & Course coordinator : 
Dr. Anushikha Singh


## Step 3: Text Chunking

In [3]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
#chunk_size=500 — each chunk will be roughly 500 characters 
#chunk_overlap=50 — each chunk shares the last 50 characters with the next chunk
chunks= text_splitter.split_documents(documents)

## Step 4: Embedding Creation

In [4]:
embeddings= HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
test_vector=embeddings.embed_query("What is RAG?")
print(f"Vector length: {len(test_vector)}")
print(test_vector[:10])  # print first 10 dimensions of the vector

C:\Users\LOQ\AppData\Local\Temp\ipykernel_5200\1505695953.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings= HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector length: 384
[-0.06957073509693146, 0.09520001709461212, 0.01602139137685299, 0.006801468785852194, -0.08840496838092804, 0.014204825274646282, 0.0540277361869812, 0.045636896044015884, -0.03192176669836044, -0.029563721269369125]


## Step 5: Vector Database (FAISS)

In [5]:
vector_store=FAISS.from_documents(chunks,embeddings)
print("Vector store created successfully")
print(f"Number of vectors in the store: {vector_store.index.ntotal}")

Vector store created successfully
Number of vectors in the store: 82


## Step 6: Query Embedding and Context Retrieval

In [6]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

query = "Layers Used to Build ConvNet?"
relevant_chunks = retriever.invoke(query)

print(f"Retrieved {len(relevant_chunks)} chunks\n")
for i, chunk in enumerate(relevant_chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content)
    print()

Retrieved 3 chunks

--- Chunk 1 ---
Layers Used to Build ConvNets
• A complete Convolution Neural Networks 
architecture is also known as covnets. A covnets is 
a sequence of layers, and every layer transforms 
one volume to another through a differentiable 
function.
Types of layers: Input layer, convolutional layer, 
pooling layer, activation layer, flattening and fully 
connected layer
4/3/2026 13

--- Chunk 2 ---
Datasets: Let’s take an example by running a covnets on of
image of dimension 32 x 32 x 3.
• Input Layers: It’s the layer in which we give input to our
model. In CNN, Generally, the input will be an image or a
sequence of images. This layer holds the raw input of the
image with width 32, height 32, and depth 3.
• Convolutional Layers: This is the layer, which is used to
extract the feature from the input dataset. It applies a set of
learnable filters known as the kernels to the input images.

--- Chunk 3 ---
• Activation Layer: By adding an activation function to the outpu

## Step 7: Answer Generation

In [7]:
llm = ChatOllama(model="llama3.1:8b", temperature=0)

def ask(question, verbose=True):
    relevant_chunks = retriever.invoke(question)
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])
    
    prompt = f"""You are answering questions using only the context below.
Give a detailed, well-explained answer — don't just list terms, explain each one briefly using the information given in the context.
Do not mention or comment on topics that were not asked about, even if they appear in the context.
If the context genuinely does not contain the answer, say "I don't know based on the provided document."

Context:
{context}

Question: {question}

Answer:"""
    
    response = llm.invoke(prompt)
    return response.content

In [8]:
print(ask("Layers Used to Build ConvNet?"))

A complete Convolutional Neural Network (ConvNet) architecture, also known as a covnet, is built using several types of layers. These layers work together to transform one volume into another through differentiable functions.

The main layers used to build a ConvNet are:

1. **Input Layer**: This layer receives the raw input data, which in this case is an image with dimensions 32 x 32 x 3 (width, height, and depth). It holds the input image before any processing takes place.
2. **Convolutional Layers**: These layers extract features from the input dataset by applying learnable filters, known as kernels, to the input images. This helps to identify patterns and edges in the image.
3. **Activation Layer**: Activation layers introduce nonlinearity into the network by applying an element-wise activation function to the output of the convolutional layer. Common activation functions include RELU (max(0, x)), Tanh, Leaky RELU, etc. This helps to prevent the "dead neuron" problem and allows the

## Step 8: Validation with Multiple Sample Questions

To confirm the pipeline generalizes beyond a single query, the system is tested with several
different questions covering different parts of the document.


In [9]:
test_questions = [
    "Layers Used to Build ConvNet?",
    "What is the difference between CNN and a fully connected network?",
    "Explain the role of the pooling layer.",
    "What activation functions are commonly used and why?",
    "What is the purpose of the convolutional layer?",
    "What is the meaning of King?",

]

validation_log = []

for q in test_questions:
    start = time.time()
    answer = ask(q, verbose=False)
    elapsed = time.time() - start
    validation_log.append({"question": q, "answer": answer, "time_sec": round(elapsed, 2)})
    print(f"Question: {q}")
    print(f"Answer: {answer}")
    print(f"Response time: {elapsed:.2f}s")
    print("-" * 80)

Question: Layers Used to Build ConvNet?
Answer: A complete Convolutional Neural Network (ConvNet) architecture, also known as a covnet, is built using a sequence of layers. The types of layers used to build a ConvNet include:

1. **Input Layer**: This layer receives the raw input data, which in this case is an image with dimensions 32 x 32 x 3. It holds the input image and prepares it for processing by the subsequent layers.

2. **Convolutional Layers**: These layers are used to extract features from the input dataset. They apply learnable filters (kernels) to the input images, allowing the network to detect patterns and features in the data.

3. **Activation Layer**: This layer adds nonlinearity to the network by applying an element-wise activation function to the output of the convolutional layer. Common activation functions include RELU, Tanh, and Leaky RELU. The activation layer does not change the dimensions of the volume; it remains 32 x 32 x 12.

4. **Pooling Layer**: This layer

## Step 9: Optimization Experiments

Two optimizations are tried against the baseline configuration (chunk_size=500, chunk_overlap=50,
pure vector search):

1. Alternate chunking configuration  to see the effect on
   retrieval .
2. Hybrid retrieval combining keyword search (BM25) with vector search (FAISS), which helps
   catch exact-term matches that pure embedding similarity can miss.


### 9.1 Chunking Comparison

In [10]:
chunk_configs = [
    {"chunk_size": 500, "chunk_overlap": 50, "label": "baseline"},
    {"chunk_size": 300, "chunk_overlap": 75, "label": "smaller"},
    {"chunk_size": 800, "chunk_overlap": 100, "label": "larger"},
]

chunking_stores = {}
chunking_retrievers = {}

for cfg in chunk_configs:
    splitter = RecursiveCharacterTextSplitter(chunk_size=cfg["chunk_size"], chunk_overlap=cfg["chunk_overlap"])
    exp_chunks = splitter.split_documents(documents)
    
    exp_store = FAISS.from_documents(exp_chunks, embeddings)
    exp_retriever = exp_store.as_retriever(search_kwargs={"k": 3})
    
    chunking_stores[cfg["label"]] = {
        "chunk_size": cfg["chunk_size"],
        "chunk_overlap": cfg["chunk_overlap"],
        "num_chunks": len(exp_chunks),
        "avg_chunk_len": round(sum(len(c.page_content) for c in exp_chunks) / len(exp_chunks), 1)
    }
    chunking_retrievers[cfg["label"]] = exp_retriever

for label, stats in chunking_stores.items():
    print(label, stats)

baseline {'chunk_size': 500, 'chunk_overlap': 50, 'num_chunks': 82, 'avg_chunk_len': 246.9}
smaller {'chunk_size': 300, 'chunk_overlap': 75, 'num_chunks': 118, 'avg_chunk_len': 195.3}
larger {'chunk_size': 800, 'chunk_overlap': 100, 'num_chunks': 67, 'avg_chunk_len': 303.9}


In [11]:
def ask_with_config(question, label):
    retriever = chunking_retrievers[label]
    relevant_chunks = retriever.invoke(question)
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])
    
    prompt = f"""You are answering questions using only the context below.
Give a detailed, well-explained answer — don't just list terms, explain each one briefly using the information given in the context.
Do not mention or comment on topics that were not asked about, even if they appear in the context.
If the context genuinely does not contain the answer, say "I don't know based on the provided document."

Context:
{context}

Question: {question}

Answer:"""
    
    response = llm.invoke(prompt)
    return response.content

In [12]:
test_query = "Layers Used to Build ConvNet?"

for label in chunking_retrievers:
    answer = ask_with_config(test_query, label)
    print(f"--- {label} (size={chunking_stores[label]['chunk_size']}, overlap={chunking_stores[label]['chunk_overlap']}) ---")
    print(answer)
    print("-" * 80)

--- baseline (size=500, overlap=50) ---
A complete Convolutional Neural Network (ConvNet) architecture, also known as a covnet, is built using a sequence of layers. The types of layers used to build a ConvNet include:

1. **Input Layer**: This layer receives the raw input data, which in this case is an image with dimensions 32 x 32 x 3. It holds the input image and prepares it for processing by the subsequent layers.

2. **Convolutional Layers**: These layers are used to extract features from the input dataset. They apply learnable filters (kernels) to the input images, allowing the network to detect patterns and features in the data.

3. **Activation Layer**: This layer adds nonlinearity to the network by applying an element-wise activation function to the output of the convolutional layer. Common activation functions include RELU, Tanh, and Leaky RELU. The activation layer does not change the dimensions of the volume; it remains 32 x 32 x 12.

4. **Pooling Layer**: This layer is peri

The smaller chunk configuration (300/75) produces more, shorter chunks, which provides finer-grained retrieval but can split related information across multiple chunks, sometimes leading to incomplete answers. The larger chunk configuration (800/100) produces fewer, longer chunks, preserving more context within each chunk but occasionally introducing extra or mixed information that reduces answer precision. The baseline configuration (500/50) provides a good balance between context preservation and retrieval accuracy, resulting in responses that are both detailed and focused.


### 9.2 Hybrid Search (Keyword + Vector)

In [13]:
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]
)

query = "Layers Used to Build ConvNet?"

vector_only_results = vector_retriever.invoke(query)
hybrid_results = hybrid_retriever.invoke(query)

print(f"Vector-only retrieved: {len(vector_only_results)} chunks")
print(f"Hybrid retrieved: {len(hybrid_results)} chunks")

Vector-only retrieved: 3 chunks
Hybrid retrieved: 5 chunks


In [14]:
def ask_hybrid(question):
    relevant_chunks = hybrid_retriever.invoke(question)
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])
    
    prompt = f"""You are answering questions using only the context below.
Give a detailed, well-explained answer — don't just list terms, explain each one briefly using the information given in the context.
Do not mention or comment on topics that were not asked about, even if they appear in the context.
If the context genuinely does not contain the answer, say "I don't know based on the provided document."

Context:
{context}

Question: {question}

Answer:"""
    
    response = llm.invoke(prompt)
    return response.content

print(ask_hybrid("Layers Used to Build ConvNet?"))

The layers used to build a Convolutional Neural Network (ConvNet) are:

1. **Input Layer**: This is the layer where we give input to our model, typically an image or a sequence of images. In this case, the input layer holds the raw input of the image with dimensions 32 x 32 and depth 3.

2. **Convolutional Layers**: These layers are used to extract features from the input dataset by applying learnable filters (kernels) to the input images.

3. **Activation Layer**: This layer adds nonlinearity to the network by applying an element-wise activation function to the output of the convolution layer, such as RELU, Tanh, or Leaky RELU. The volume remains unchanged, and the output volume has dimensions 32 x 32 x 12.

4. **Pooling Layer**: Periodically inserted in the ConvNet connections, this layer allows gradients to bypass some layers and directly flow to deeper layers.

5. **Flattening and Fully Connected Layer**: Although not explicitly mentioned as part of the ConvNet architecture, these 

Combining BM25 keyword search with the FAISS vector retriever allows the system to pick up
exact term matches (e.g. specific layer names) that a pure embedding search can sometimes rank
lower, while still keeping the semantic matching strength of vector search.


## Step 10: System Metrics Report

In [15]:
metrics_report = {
    "document_source": os.path.basename(pdf_path),
    "num_pages_loaded": len(documents),
    "chunking": {
        "chunk_size": 500,
        "chunk_overlap": 50,
        "num_chunks": len(chunks)
    },
    "chunking_configs_tested": [
        {"chunk_size": chunking_stores[l]["chunk_size"], "chunk_overlap": chunking_stores[l]["chunk_overlap"], "num_chunks": chunking_stores[l]["num_chunks"]}
        for l in chunking_stores
    ],
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_dimensions": len(test_vector),
    "vector_store": {
        "type": "FAISS",
        "num_vectors": vector_store.index.ntotal
    },
    "retrieval": {
        "top_k": 3,
        "strategies_tested": ["vector-only (FAISS)", "hybrid (BM25 + FAISS, weights 0.4/0.6)"]
    },
    "llm": {
        "provider": "Ollama",
        "model": "llama3.1:8b",
        "temperature": 0
    },
    "validation_questions_tested": len(test_questions)
}

for k, v in metrics_report.items():
    print(f"{k}: {v}")

document_source: unit4 dl.pdf
num_pages_loaded: 61
chunking: {'chunk_size': 500, 'chunk_overlap': 50, 'num_chunks': 82}
chunking_configs_tested: [{'chunk_size': 500, 'chunk_overlap': 50, 'num_chunks': 82}, {'chunk_size': 300, 'chunk_overlap': 75, 'num_chunks': 118}, {'chunk_size': 800, 'chunk_overlap': 100, 'num_chunks': 67}]
embedding_model: sentence-transformers/all-MiniLM-L6-v2
embedding_dimensions: 384
vector_store: {'type': 'FAISS', 'num_vectors': 82}
retrieval: {'top_k': 3, 'strategies_tested': ['vector-only (FAISS)', 'hybrid (BM25 + FAISS, weights 0.4/0.6)']}
llm: {'provider': 'Ollama', 'model': 'llama3.1:8b', 'temperature': 0}
validation_questions_tested: 6
